# From a dispel4py Sensor Workflow to an Agentic AI Workflow

**A tiny hands-on tutorial using dispel4py and an OpenAI LLM**

This notebook starts from the structure of the public dispel4py `SensorWorkflow.py` example:

```text
read sensor data
        ↓
normalise temperature
        ↓
detect anomalies
        ↓
print alerts
        ↓
aggregate readings
```

We then extend the idea into an **agentic AI sensor workflow** in which one stateful dispel4py Processing Element can inspect evidence, call bounded tools, observe their results, and choose a final action.

This notebook uses:

- **dispel4py** for the stream-based workflow;
- the **OpenAI Python SDK** for the LLM and function calling;
- ordinary Python for deterministic rules and tools;
- **no LangGraph and no LangChain**.


## What will we build?

The tutorial has two workflows.

### Workflow A: deterministic sensor dataflow

```text
ReadSensorDataPE
        ↓
NormalizeDataPE
        ↓
AnomalyDetectionPE
        ↓
AlertingPE
        ↓
AggregateDataPE
```

This follows the original dispel4py sensor example. It is a useful stream-processing workflow, but it is not yet an LLM agent.

### Workflow B: agentic AI sensor workflow

```text
ReadSensorDataPE
        ↓
NormalizeDataPE
        ↓
DeterministicPrecheckPE
   ├── hard rule applies ──────────────────────┐
   │                                           ↓
   └── richer interpretation needed → LLMSensorAgentPE
                                           │
                                           ├── inspect previous readings
                                           ├── compare neighbouring sensors
                                           ├── request another measurement
                                           ├── create maintenance ticket
                                           ├── notify operator
                                           └── escalate to human
                                           ↓
                                     DecisionMergePE
                                           ↓
                                     ActionExecutorPE
                                           ↓
                                     ResultWriterPE
```

Most PEs remain ordinary deterministic workflow components. The strongly agentic component is `LLMSensorAgentPE`.


## When is a dispel4py workflow agentic?

A Processing Element is not automatically an agent.

A component becomes agent-like when it has a role, access to relevant context or state, actions it can select, feedback from those actions, and a goal it continues working towards.

In the second workflow, `LLMSensorAgentPE` is agentic because it:

1. receives the current sensor event;
2. retains recent sensor history and neighbouring readings;
3. decides whether it needs more evidence;
4. chooses and calls one or more permitted Python tools;
5. observes the tool results;
6. continues the loop;
7. submits one bounded final decision.

The tools are not agents. They are actions available to the agent.


## 1. Install dispel4py and the OpenAI SDK

The uploaded dispel4py testing notebook installs the package as `stream-d4py`. The cell below also shows an optional GitHub installation for using the current `StreamingFlow/d4py` main branch.

Run **one** dispel4py installation route, not both.


In [ ]:
# Recommended for this tutorial: current development version from GitHub
!pip install -qU mpi4py openai
!pip install -qU "git+https://github.com/StreamingFlow/d4py.git@main"

# Stable PyPI alternative used in the existing testing notebook:
# !pip install -qU mpi4py openai stream-d4py


## 2. Create a small sensor dataset

The data contains repeated readings, neighbouring sensors, a low-battery event, packet loss, possible moisture, and possible calibration drift.

All examples are synthetic. No real device is controlled.


In [ ]:
    import json

    sensor_data = [
    {
        "sensor_id": "sensor-001",
        "zone": "north",
        "timestamp": "2026-08-06T09:00:00Z",
        "temperature": 21.8,
        "humidity": 42.0,
        "battery": 83,
        "diagnostic_note": "Routine reading. Signal is stable."
    },
    {
        "sensor_id": "sensor-002",
        "zone": "north",
        "timestamp": "2026-08-06T09:01:00Z",
        "temperature": 22.0,
        "humidity": 43.0,
        "battery": 79,
        "diagnostic_note": "Routine reading from a neighbouring sensor."
    },
    {
        "sensor_id": "sensor-001",
        "zone": "north",
        "timestamp": "2026-08-06T09:10:00Z",
        "temperature": 22.2,
        "humidity": 44.0,
        "battery": 82,
        "diagnostic_note": "Two packets were lost after a brief network interruption. The connection has returned."
    },
    {
        "sensor_id": "sensor-003",
        "zone": "south",
        "timestamp": "2026-08-06T09:12:00Z",
        "temperature": 20.9,
        "humidity": 46.0,
        "battery": 5,
        "diagnostic_note": "Reading is plausible but battery is very low."
    },
    {
        "sensor_id": "sensor-004",
        "zone": "south",
        "timestamp": "2026-08-06T09:14:00Z",
        "temperature": 19.7,
        "humidity": 88.0,
        "battery": 72,
        "diagnostic_note": "A technician heard intermittent buzzing and is unsure whether moisture entered the enclosure."
    },
    {
        "sensor_id": "sensor-002",
        "zone": "north",
        "timestamp": "2026-08-06T09:16:00Z",
        "temperature": 37.5,
        "humidity": 43.0,
        "battery": 78,
        "diagnostic_note": "Sudden temperature increase. The sensor reports that calibration may have drifted."
    }
]

    with open("sensor_data_agentic.json", "w", encoding="utf-8") as handle:
        json.dump(sensor_data, handle, indent=2)

    print("Created sensor_data_agentic.json")
    print("Number of readings:", len(sensor_data))


# Part I — The original deterministic sensor pattern

We first reproduce the basic idea of the public `SensorWorkflow.py` example.

The workflow reads the JSON file, normalises each temperature, compares it with a running mean, prints alerts, and prints an average after every five readings.

The behaviour is predetermined:

```text
input arrives
    ↓
run the next PE
    ↓
apply its fixed Python function
```

This is a scientific streaming workflow, but the PEs do not choose tools or gather additional evidence.


In [ ]:
    %%writefile sensor_workflow_baseline.py
    from dispel4py.base import ConsumerPE, IterativePE, ProducerPE
from dispel4py.workflow_graph import WorkflowGraph
import json
import numpy as np


class ReadSensorDataPE(ProducerPE):
    """Read a JSON file and emit one sensor record at a time."""

    def __init__(self):
        super().__init__()

    def _process(self, inputs):
        file_path = inputs["input"]
        with open(file_path, "r", encoding="utf-8") as handle:
            data = json.load(handle)

        for record in data:
            self.write(
                "output",
                {
                    "sensor_id": record["sensor_id"],
                    "zone": record["zone"],
                    "timestamp": record["timestamp"],
                    "temperature": float(record["temperature"]),
                    "humidity": float(record["humidity"]),
                    "battery": int(record["battery"]),
                    "diagnostic_note": record["diagnostic_note"],
                },
            )


class NormalizeDataPE(IterativePE):
    """Normalise temperature to a 0-1 scale for an expected 0-40°C range."""

    def __init__(self):
        super().__init__()

    def _process(self, data):
        result = dict(data)
        result["normalized_temperature"] = result["temperature"] / 40.0
        return result


class AnomalyDetectionPE(IterativePE):
    """
    Compare each normalised temperature with the running mean.

    This follows the spirit of the original SensorWorkflow.py example.
    """

    def __init__(self, threshold=0.10):
        super().__init__()
        self.threshold = threshold
        self.temperatures = []

    def _process(self, data):
        result = dict(data)
        current = result["normalized_temperature"]

        self.temperatures.append(current)
        running_mean = float(np.mean(self.temperatures))

        result["running_mean"] = running_mean
        result["anomaly"] = abs(current - running_mean) > self.threshold
        return result


class AlertingPE(IterativePE):
    """Print a deterministic alert when the anomaly flag is true."""

    def __init__(self):
        super().__init__()

    def _process(self, data):
        if data.get("anomaly", False):
            print(
                "BASELINE ANOMALY:",
                data["sensor_id"],
                data["timestamp"],
                f"temperature={data['temperature']}",
            )
        return data


class AggregateDataPE(ConsumerPE):
    """Print an average after every five readings."""

    def __init__(self):
        super().__init__()
        self.temperatures = []

    def _process(self, data):
        self.temperatures.append(data["temperature"])

        if len(self.temperatures) == 5:
            print(
                "BASELINE AVERAGE OF 5 READINGS:",
                float(np.mean(self.temperatures)),
            )
            self.temperatures = []


read = ReadSensorDataPE()
read.name = "read"

normalize = NormalizeDataPE()
detect = AnomalyDetectionPE(threshold=0.10)
alert = AlertingPE()
aggregate = AggregateDataPE()

graph = WorkflowGraph()
graph.connect(read, "output", normalize, "input")
graph.connect(normalize, "output", detect, "input")
graph.connect(detect, "output", alert, "input")
graph.connect(alert, "output", aggregate, "input")


## 3. Run the baseline workflow

The `-d` argument supplies the file path to the PE named `read`.


In [ ]:
!dispel4py simple sensor_workflow_baseline.py       -d '{"read": [{"input": "/content/sensor_data_agentic.json"}]}'


## What is missing from the baseline?

The baseline can detect a numerical deviation, but it cannot:

- inspect previous readings on demand;
- compare neighbouring sensors;
- distinguish temporary packet loss from likely equipment damage;
- request a fresh measurement;
- create a maintenance ticket;
- notify an operator;
- escalate an ambiguous case to a human;
- decide which of those actions is most appropriate.

We now retain the deterministic dataflow, but add one bounded LLM-powered agent PE.


# Part II — Add a tool-using LLM Sensor Agent

The next workflow deliberately uses a **hybrid design**.

Hard numerical rules stay deterministic. For example:

```text
battery below 10% → maintenance
implausible temperature → human review
```

Events not resolved by those rules are sent to `LLMSensorAgentPE`.

The LLM does not directly control equipment. It can only call Python functions defined by the programmer, and all external effects are simulated.


## The agent's tools

The agent may call:

- `inspect_previous_readings`
- `compare_neighbouring_sensors`
- `request_another_measurement`
- `create_maintenance_ticket`
- `notify_operator`
- `escalate_to_human`
- `submit_final_decision`

Its final decision must be one of:

```text
accept | retry | maintenance | notify_operator | human_review
```

This is what makes the example more than a one-shot LLM classification: the model can gather evidence, act, observe the tool result, and continue before finishing.


## 4. Add the OpenAI API key securely

The key is entered without being displayed and is stored only in the current Colab runtime.

The default model is `gpt-5-mini`. You can change `OPENAI_MODEL` if your API project uses another tool-capable model.


In [ ]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass(
        "Paste your OpenAI API key: "
    )

os.environ.setdefault("OPENAI_MODEL", "gpt-5-mini")
print("Model:", os.environ["OPENAI_MODEL"])


## 5. Define the complete agentic dispel4py workflow

The workflow writes final records to `agentic_sensor_results.jsonl`.

The agent maintains state inside its PE instance:

- recent readings for each sensor;
- the latest reading from each neighbouring sensor;
- simulated measurement requests;
- simulated maintenance tickets;
- simulated operator notifications;
- simulated human-review cases.

The simple mapping is used because it processes the events sequentially, making the state and tool trace easy to understand.


In [ ]:
    %%writefile sensor_workflow_agentic.py
    from collections import defaultdict, deque
from datetime import datetime, timezone
from dispel4py.base import ConsumerPE, GenericPE, IterativePE, ProducerPE
from dispel4py.workflow_graph import WorkflowGraph
from openai import OpenAI
import json
import os


INPUT_FILE = "sensor_data_agentic.json"
OUTPUT_FILE = "agentic_sensor_results.jsonl"


def utc_now():
    return datetime.now(timezone.utc).isoformat()


class ReadSensorDataPE(ProducerPE):
    """Read the JSON file and emit one enriched event at a time."""

    def __init__(self):
        super().__init__()

    def _process(self, inputs):
        file_path = inputs["input"]

        with open(file_path, "r", encoding="utf-8") as handle:
            records = json.load(handle)

        for record in records:
            event = {
                "sensor_id": record["sensor_id"],
                "zone": record["zone"],
                "timestamp": record["timestamp"],
                "temperature": float(record["temperature"]),
                "humidity": float(record["humidity"]),
                "battery": int(record["battery"]),
                "diagnostic_note": record["diagnostic_note"],
                "audit": [
                    f"{utc_now()} ReadSensorDataPE emitted "
                    f"{record['sensor_id']}."
                ],
            }
            self.write("output", event)


class NormalizeDataPE(IterativePE):
    """Retain the familiar deterministic normalisation step."""

    def __init__(self):
        super().__init__()

    def _process(self, data):
        result = dict(data)
        result["normalized_temperature"] = result["temperature"] / 40.0
        result["audit"] = list(result["audit"]) + [
            f"{utc_now()} NormalizeDataPE normalised the temperature."
        ]
        return result


class DeterministicPrecheckPE(GenericPE):
    """
    Apply hard rules before the LLM is considered.

    This PE is not the LLM agent. It is a deterministic safety gate.
    """

    def __init__(self):
        super().__init__()
        self._add_input("input")
        self._add_output("resolved")
        self._add_output("needs_agent")

    def _process(self, inputs):
        event = dict(inputs["input"])
        audit = list(event["audit"])

        if event["battery"] < 10:
            audit.append(
                f"{utc_now()} DeterministicPrecheckPE selected maintenance "
                "because battery was below 10%."
            )
            return {
                "resolved": {
                    **event,
                    "action": "maintenance",
                    "reason": (
                        f"Battery is {event['battery']}%, below the fixed "
                        "10% maintenance threshold."
                    ),
                    "decision_source": "deterministic_rule",
                    "tool_trace": [],
                    "audit": audit,
                }
            }

        if event["temperature"] < -40 or event["temperature"] > 85:
            audit.append(
                f"{utc_now()} DeterministicPrecheckPE selected human_review "
                "because temperature was outside the fixed plausible range."
            )
            return {
                "resolved": {
                    **event,
                    "action": "human_review",
                    "reason": (
                        f"Temperature {event['temperature']}°C is outside "
                        "the fixed -40°C to 85°C range."
                    ),
                    "decision_source": "deterministic_rule",
                    "tool_trace": [],
                    "audit": audit,
                }
            }

        audit.append(
            f"{utc_now()} No hard rule applied; event sent to "
            "LLMSensorAgentPE."
        )
        return {"needs_agent": {**event, "audit": audit}}


class LLMSensorAgentPE(IterativePE):
    """
    A stateful, tool-using agent implemented inside one dispel4py PE.

    The agent can inspect evidence, invoke bounded Python tools, observe their
    results, and continue until it submits a final decision.
    """

    FINAL_ACTIONS = {
        "accept",
        "retry",
        "maintenance",
        "notify_operator",
        "human_review",
    }

    EVIDENCE_TOOLS = {
        "inspect_previous_readings",
        "compare_neighbouring_sensors",
        "request_another_measurement",
    }

    def __init__(
        self,
        neighbour_map,
        model="gpt-5-mini",
        history_size=5,
        max_rounds=6,
    ):
        super().__init__()
        self.neighbour_map = neighbour_map
        self.model = model
        self.history_size = history_size
        self.max_rounds = max_rounds

        self.client = None
        self.history = defaultdict(lambda: deque(maxlen=history_size))
        self.latest_by_sensor = {}

        # These collections simulate external systems.
        self.measurement_requests = []
        self.maintenance_tickets = []
        self.operator_notifications = []
        self.human_review_queue = []

        self.tools = self._build_tools()

    def preprocess(self):
        if not os.environ.get("OPENAI_API_KEY"):
            raise RuntimeError(
                "OPENAI_API_KEY is missing. Run the API-key cell first."
            )
        self.client = OpenAI()

    def _build_tools(self):
        return [
            {
                "type": "function",
                "name": "inspect_previous_readings",
                "description": (
                    "Inspect up to five recent readings from the current "
                    "sensor before deciding."
                ),
                "parameters": {
                    "type": "object",
                    "properties": {
                        "sensor_id": {"type": "string"},
                        "limit": {
                            "type": "integer",
                            "minimum": 1,
                            "maximum": 5,
                        },
                    },
                    "required": ["sensor_id", "limit"],
                    "additionalProperties": False,
                },
                "strict": True,
            },
            {
                "type": "function",
                "name": "compare_neighbouring_sensors",
                "description": (
                    "Compare the current reading with the most recent readings "
                    "from configured neighbouring sensors."
                ),
                "parameters": {
                    "type": "object",
                    "properties": {
                        "sensor_id": {"type": "string"},
                    },
                    "required": ["sensor_id"],
                    "additionalProperties": False,
                },
                "strict": True,
            },
            {
                "type": "function",
                "name": "request_another_measurement",
                "description": (
                    "Create a simulated request for another measurement when "
                    "a temporary communication or sampling issue is suspected."
                ),
                "parameters": {
                    "type": "object",
                    "properties": {
                        "sensor_id": {"type": "string"},
                        "reason": {"type": "string"},
                    },
                    "required": ["sensor_id", "reason"],
                    "additionalProperties": False,
                },
                "strict": True,
            },
            {
                "type": "function",
                "name": "create_maintenance_ticket",
                "description": (
                    "Create a simulated maintenance ticket for likely "
                    "hardware, battery, calibration, or enclosure problems."
                ),
                "parameters": {
                    "type": "object",
                    "properties": {
                        "sensor_id": {"type": "string"},
                        "reason": {"type": "string"},
                        "priority": {
                            "type": "string",
                            "enum": ["low", "medium", "high"],
                        },
                    },
                    "required": ["sensor_id", "reason", "priority"],
                    "additionalProperties": False,
                },
                "strict": True,
            },
            {
                "type": "function",
                "name": "notify_operator",
                "description": (
                    "Create a simulated operator notification for an event "
                    "that should be visible to operations staff."
                ),
                "parameters": {
                    "type": "object",
                    "properties": {
                        "sensor_id": {"type": "string"},
                        "message": {"type": "string"},
                        "severity": {
                            "type": "string",
                            "enum": ["info", "warning", "critical"],
                        },
                    },
                    "required": ["sensor_id", "message", "severity"],
                    "additionalProperties": False,
                },
                "strict": True,
            },
            {
                "type": "function",
                "name": "escalate_to_human",
                "description": (
                    "Create a simulated human-review item for ambiguous, "
                    "contradictory, or safety-relevant cases."
                ),
                "parameters": {
                    "type": "object",
                    "properties": {
                        "sensor_id": {"type": "string"},
                        "reason": {"type": "string"},
                    },
                    "required": ["sensor_id", "reason"],
                    "additionalProperties": False,
                },
                "strict": True,
            },
            {
                "type": "function",
                "name": "submit_final_decision",
                "description": (
                    "Finish the agent loop by submitting exactly one final "
                    "bounded action and a concise reason."
                ),
                "parameters": {
                    "type": "object",
                    "properties": {
                        "action": {
                            "type": "string",
                            "enum": [
                                "accept",
                                "retry",
                                "maintenance",
                                "notify_operator",
                                "human_review",
                            ],
                        },
                        "reason": {"type": "string"},
                    },
                    "required": ["action", "reason"],
                    "additionalProperties": False,
                },
                "strict": True,
            },
        ]

    # ----------------------- deterministic tools -----------------------

    def inspect_previous_readings(self, sensor_id, limit, current):
        previous = list(self.history[sensor_id])[-limit:]
        return {
            "sensor_id": sensor_id,
            "count": len(previous),
            "previous_readings": previous,
            "current_reading_excluded": True,
        }

    def compare_neighbouring_sensors(self, sensor_id, current):
        neighbours = []

        for neighbour_id in self.neighbour_map.get(sensor_id, []):
            latest = self.latest_by_sensor.get(neighbour_id)
            if latest:
                neighbours.append(
                    {
                        "sensor_id": neighbour_id,
                        "timestamp": latest["timestamp"],
                        "temperature": latest["temperature"],
                        "humidity": latest["humidity"],
                    }
                )

        differences = [
            {
                "sensor_id": item["sensor_id"],
                "temperature_difference": round(
                    current["temperature"] - item["temperature"], 2
                ),
                "humidity_difference": round(
                    current["humidity"] - item["humidity"], 2
                ),
            }
            for item in neighbours
        ]

        return {
            "sensor_id": sensor_id,
            "neighbours_found": len(neighbours),
            "neighbour_readings": neighbours,
            "differences_from_current": differences,
        }

    def request_another_measurement(self, sensor_id, reason, current):
        request = {
            "request_id": f"RM-{len(self.measurement_requests) + 1:04d}",
            "sensor_id": sensor_id,
            "reason": reason,
            "created_at": utc_now(),
            "status": "simulated",
        }
        self.measurement_requests.append(request)
        return request

    def create_maintenance_ticket(
        self, sensor_id, reason, priority, current
    ):
        ticket = {
            "ticket_id": f"MT-{len(self.maintenance_tickets) + 1:04d}",
            "sensor_id": sensor_id,
            "priority": priority,
            "reason": reason,
            "created_at": utc_now(),
            "status": "simulated",
        }
        self.maintenance_tickets.append(ticket)
        return ticket

    def notify_operator(self, sensor_id, message, severity, current):
        notification = {
            "notification_id": (
                f"OP-{len(self.operator_notifications) + 1:04d}"
            ),
            "sensor_id": sensor_id,
            "severity": severity,
            "message": message,
            "created_at": utc_now(),
            "status": "simulated",
        }
        self.operator_notifications.append(notification)
        return notification

    def escalate_to_human(self, sensor_id, reason, current):
        review = {
            "review_id": f"HR-{len(self.human_review_queue) + 1:04d}",
            "sensor_id": sensor_id,
            "reason": reason,
            "created_at": utc_now(),
            "status": "simulated",
        }
        self.human_review_queue.append(review)
        return review

    def _execute_tool(self, name, arguments, current):
        tool_map = {
            "inspect_previous_readings": self.inspect_previous_readings,
            "compare_neighbouring_sensors": (
                self.compare_neighbouring_sensors
            ),
            "request_another_measurement": (
                self.request_another_measurement
            ),
            "create_maintenance_ticket": (
                self.create_maintenance_ticket
            ),
            "notify_operator": self.notify_operator,
            "escalate_to_human": self.escalate_to_human,
        }

        if name not in tool_map:
            raise ValueError(f"Tool is not permitted: {name}")

        return tool_map[name](current=current, **arguments)

    # -------------------------- agent loop -----------------------------

    def _process(self, event):
        if self.client is None:
            self.preprocess()

        current = dict(event)
        audit = list(current["audit"])
        tool_trace = []
        final_decision = None

        instructions = """
You are LLMSensorAgent, a bounded IoT sensor agent inside a dispel4py
streaming workflow.

Your goal is to assess one sensor event safely.

You may gather evidence by inspecting previous readings, comparing
neighbouring sensors, or requesting another measurement. You may also create
a maintenance ticket, notify an operator, or escalate to a human.

Rules:
- Never invent or modify measurements.
- Prefer deterministic tool evidence over speculation.
- Use at least one evidence-gathering tool before accepting an event that
  contains a warning, uncertainty, packet loss, moisture, buzzing, drift, or
  another possible fault.
- Create a maintenance ticket only for likely equipment, enclosure, battery,
  or calibration problems.
- Escalate ambiguous or safety-relevant cases to a human.
- Finish by calling submit_final_decision.
- If you request another measurement, the final action should normally be
  retry.
- If you create a maintenance ticket, the final action must be maintenance.
- If you escalate to a human, the final action must be human_review.
"""

        first_input = (
            "Assess this sensor event:\n"
            + json.dumps(
                {
                    "sensor_id": current["sensor_id"],
                    "zone": current["zone"],
                    "timestamp": current["timestamp"],
                    "temperature": current["temperature"],
                    "normalized_temperature": (
                        current["normalized_temperature"]
                    ),
                    "humidity": current["humidity"],
                    "battery": current["battery"],
                    "diagnostic_note": current["diagnostic_note"],
                },
                indent=2,
            )
        )

        try:
            response = self.client.responses.create(
                model=self.model,
                instructions=instructions,
                input=first_input,
                tools=self.tools,
                tool_choice="auto",
            )

            for round_number in range(1, self.max_rounds + 1):
                calls = [
                    item
                    for item in response.output
                    if getattr(item, "type", None) == "function_call"
                ]

                if not calls:
                    raise RuntimeError(
                        "The model returned no tool call or final decision."
                    )

                outputs = []

                for call in calls:
                    arguments = json.loads(call.arguments)

                    if call.name == "submit_final_decision":
                        action = arguments["action"]
                        reason = arguments["reason"]

                        if action not in self.FINAL_ACTIONS:
                            raise ValueError(
                                "Final action was outside the permitted set."
                            )

                        final_decision = {
                            "action": action,
                            "reason": reason,
                        }
                        audit.append(
                            f"{utc_now()} LLMSensorAgentPE submitted "
                            f"{action}: {reason}"
                        )
                        break

                    tool_result = self._execute_tool(
                        call.name,
                        arguments,
                        current,
                    )

                    tool_trace.append(
                        {
                            "round": round_number,
                            "tool": call.name,
                            "arguments": arguments,
                            "result": tool_result,
                        }
                    )
                    audit.append(
                        f"{utc_now()} LLMSensorAgentPE called "
                        f"{call.name}."
                    )

                    outputs.append(
                        {
                            "type": "function_call_output",
                            "call_id": call.call_id,
                            "output": json.dumps(tool_result),
                        }
                    )

                if final_decision is not None:
                    break

                response = self.client.responses.create(
                    model=self.model,
                    instructions=instructions,
                    previous_response_id=response.id,
                    input=outputs,
                    tools=self.tools,
                    tool_choice="auto",
                )

            if final_decision is None:
                raise RuntimeError(
                    "Maximum agent rounds reached without a final decision."
                )

            result = {
                **current,
                **final_decision,
                "decision_source": "openai_tool_using_agent",
                "tool_trace": tool_trace,
                "audit": audit,
            }

        except Exception as exc:
            reason = (
                f"Agent or API failure: {type(exc).__name__}: {exc}"
            )
            review = self.escalate_to_human(
                sensor_id=current["sensor_id"],
                reason=reason,
                current=current,
            )
            tool_trace.append(
                {
                    "round": None,
                    "tool": "escalate_to_human",
                    "arguments": {
                        "sensor_id": current["sensor_id"],
                        "reason": reason,
                    },
                    "result": review,
                    "fallback": True,
                }
            )
            audit.append(
                f"{utc_now()} Safe fallback sent the event to human review."
            )

            result = {
                **current,
                "action": "human_review",
                "reason": reason,
                "decision_source": "agent_error_fallback",
                "tool_trace": tool_trace,
                "audit": audit,
            }

        # Update memory only after the decision, so "previous" excludes current.
        self.history[current["sensor_id"]].append(
            {
                "timestamp": current["timestamp"],
                "temperature": current["temperature"],
                "humidity": current["humidity"],
                "battery": current["battery"],
                "diagnostic_note": current["diagnostic_note"],
                "final_action": result["action"],
            }
        )
        self.latest_by_sensor[current["sensor_id"]] = current

        return result


class DecisionMergePE(GenericPE):
    """Merge the deterministic and LLM-agent branches."""

    def __init__(self):
        super().__init__()
        self._add_input("deterministic")
        self._add_input("agent")
        self._add_output("output")

    def _process(self, inputs):
        if "deterministic" in inputs:
            return {"output": inputs["deterministic"]}
        if "agent" in inputs:
            return {"output": inputs["agent"]}
        return None


class ActionExecutorPE(IterativePE):
    """
    Execute the selected bounded action.

    This tutorial records simulated actions rather than controlling devices.
    """

    def __init__(self):
        super().__init__()

    def _process(self, decision):
        outcomes = {
            "accept": "Reading stored as valid.",
            "retry": "A follow-up measurement was requested.",
            "maintenance": "Maintenance workflow initiated.",
            "notify_operator": "Operator notification recorded.",
            "human_review": "Case placed in human-review queue.",
        }

        action = decision["action"]

        if action not in outcomes:
            raise ValueError(f"Unsupported final action: {action}")

        result = {
            **decision,
            "outcome": outcomes[action],
            "audit": list(decision["audit"])
            + [
                f"{utc_now()} ActionExecutorPE: "
                f"{outcomes[action]}"
            ],
        }

        print(
            "AGENTIC RESULT:",
            result["sensor_id"],
            "->",
            result["action"],
            f"({result['decision_source']})",
        )

        return result


class ResultWriterPE(ConsumerPE):
    """Write every final event to a JSON Lines file."""

    def __init__(self, output_file=OUTPUT_FILE):
        super().__init__()
        self.output_file = output_file

    def _process(self, result):
        with open(self.output_file, "a", encoding="utf-8") as handle:
            handle.write(json.dumps(result) + "\n")


NEIGHBOUR_MAP = {
    "sensor-001": ["sensor-002"],
    "sensor-002": ["sensor-001"],
    "sensor-003": ["sensor-004"],
    "sensor-004": ["sensor-003"],
}


read = ReadSensorDataPE()
read.name = "read"

normalize = NormalizeDataPE()
precheck = DeterministicPrecheckPE()
agent = LLMSensorAgentPE(
    neighbour_map=NEIGHBOUR_MAP,
    model=os.environ.get("OPENAI_MODEL", "gpt-5-mini"),
)
merge = DecisionMergePE()
execute = ActionExecutorPE()
write_results = ResultWriterPE()

graph = WorkflowGraph()
graph.connect(read, "output", normalize, "input")
graph.connect(normalize, "output", precheck, "input")
graph.connect(precheck, "resolved", merge, "deterministic")
graph.connect(precheck, "needs_agent", agent, "input")
graph.connect(agent, "output", merge, "agent")
graph.connect(merge, "output", execute, "input")
graph.connect(execute, "output", write_results, "input")


## 6. Run the agentic workflow

Delete any previous results first, then execute the graph.

The low-battery event should be resolved by the deterministic precheck without an LLM call. Other events may cause the agent to inspect history, compare neighbours, request another measurement, create a ticket, notify an operator, or escalate.


In [ ]:
import os

if os.path.exists("agentic_sensor_results.jsonl"):
    os.remove("agentic_sensor_results.jsonl")

!dispel4py simple sensor_workflow_agentic.py       -d '{"read": [{"input": "/content/sensor_data_agentic.json"}]}'


## 7. Inspect the final decisions and tool calls

Each output record includes:

- `action`
- `reason`
- `decision_source`
- `tool_trace`
- `audit`
- `outcome`

LLM behaviour can vary slightly. The available tool names and final actions cannot vary because they are constrained by schemas and checked again in Python.


In [ ]:
import json

results = []

with open(
    "agentic_sensor_results.jsonl",
    "r",
    encoding="utf-8",
) as handle:
    for line in handle:
        if line.strip():
            results.append(json.loads(line))

for result in results:
    print(
        f"\n{result['sensor_id']} at {result['timestamp']}"
        f"\n  action: {result['action']}"
        f"\n  source: {result['decision_source']}"
        f"\n  reason: {result['reason']}"
    )

    if result["tool_trace"]:
        print("  tools:")
        for item in result["tool_trace"]:
            print("   -", item["tool"])
    else:
        print("  tools: none")


## 8. Inspect one complete audit trail

The audit trail shows how dispel4py moved the event through the graph and how the agent interacted with its tools.


In [ ]:
# Choose the first result that used at least one tool.
selected = next(
    (item for item in results if item["tool_trace"]),
    results[0],
)

print("Selected event:", selected["sensor_id"], selected["timestamp"])

print("\nAudit:")
for line in selected["audit"]:
    print(" -", line)

print("\nTool trace:")
for item in selected["tool_trace"]:
    print(json.dumps(item, indent=2))


# What exactly is agentic here?

The complete workflow is agentic because it contains a goal-directed component that can select and use actions.

```text
deterministic workflow components
    ReadSensorDataPE
    NormalizeDataPE
    DeterministicPrecheckPE
    DecisionMergePE
    ActionExecutorPE
    ResultWriterPE

agentic component
    LLMSensorAgentPE
```

`LLMSensorAgentPE` is not agentic merely because it contains an LLM. It is agentic because it combines:

- a role and goal;
- state and context;
- bounded tool selection;
- feedback from tool results;
- an iterative loop;
- a stopping action;
- safe fallback to human review.

A deterministic PE could also be agentic if it had comparable goal-directed, stateful action-selection behaviour. An LLM is not a requirement for agentic behaviour; here it is useful for interpreting ambiguous diagnostic language.


# Where dispel4py fits

dispel4py provides the stream-based dataflow:

- PEs define processing stages;
- streams connect outputs to inputs;
- the mapping executes the abstract graph;
- the same scientific workflow structure can be adapted to different execution environments.

The LLM agent is hosted inside one stateful PE. Therefore, this example shows that agentic AI does not require LangGraph or LangChain.

```text
dispel4py → streaming workflow and execution
OpenAI LLM → bounded interpretation and tool selection
Python tools → deterministic evidence and actions
human review → cases that should not be fully automated
```


# Important limitations

This is a teaching example, not a production control system.

- The sensor data is synthetic.
- Tickets, notifications, and measurements are simulated.
- The LLM may make a different bounded choice on another run.
- A real deployment would require authentication, persistent state, monitoring, retries, evaluation, domain-reviewed policies, and human approval.
- Safety-critical actuators should not be controlled directly by the model.
- The simple mapping is used for clarity. A stateful agent PE would need careful partitioning and state management under parallel mappings.

The key design principle is to keep hard rules and safety checks deterministic, while using the LLM only where interpretation and flexible tool selection are genuinely useful.


# References

- [StreamingFlow/d4py](https://github.com/StreamingFlow/d4py)
- [StreamingFlow/d4py_workflows — SensorWorkflow.py](https://github.com/StreamingFlow/d4py_workflows/blob/main/others/SensorWorkflow.py)
- [OpenAI Python SDK](https://github.com/openai/openai-python)
- [OpenAI function calling](https://platform.openai.com/docs/guides/function-calling)
